# 03 — Pilot đo tỷ lệ nhãn và chia tập dữ liệu

Trước khi chia Train / Validation / Hold-out, tôi cần biết mỗi nhãn chiếm bao nhiêu phần trong dữ liệu. Kích thước Hold-out
được tính theo lớp hiếm nhất (N ≈ m / p_rare với m = 80, trần 1.000 — Q8), nên nếu đoán sai tỷ lệ thì Hold-out có thể có quá ít
review FEATURE hoặc UX để đánh giá đáng tin. Calibration cho tôi một ước lượng thô (trên 150 review ngẫu nhiên chỉ có khoảng 5%
BUG, 5% FEATURE, 7% UX), nhưng con số đó quá nhỏ để dựa vào. Vì vậy bước đầu tiên của notebook này là một pilot 400 review.

Một vài điểm về thiết kế:

- Pilot được rút **ngẫu nhiên thuần** (seed 42) từ corpus Shopee trung gian, sau khi loại 180 review calibration. Tôi không chọn
  theo từ khóa, không bỏ review trùng nội dung, không lọc theo số sao, để tỷ lệ đo được phản ánh đúng dữ liệu (corpus gần như toàn
  bộ review 12 tháng — amendment v1.5).
- Pilot là một phần của Hold-out (Q9). Vì vậy các file pilot thuộc diện khóa của `HOLDOUT_LOCK.md`: hai người gán độc lập bằng
  guideline v1.2 đã đóng băng, ca lệch cũng chốt theo đúng guideline đó, và tôi không dùng pilot để sửa guideline hay làm ví dụ.
- Bảng gán chỉ có `review_id` và nội dung review; số sao, ngày, phiên bản app được ẩn (Q17).

Ở nhịp này tôi mới rút mẫu và chuẩn bị phần đo. Việc tính kích thước Hold-out và chia tập sẽ làm sau, khi đã có tỷ lệ từ pilot.
Phần xử lý nằm trong `src/sampling/pilot.py`.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
import pandas as pd
from src.utils.kaggle_paths import input_dir
from src.utils.seeds import set_seed
from src.preprocessing.clean import load_jsonl
from src.annotation.calibration import export_blank
from src.annotation.agreement import merge, agreement, disagreements_by_pair, build_finalize_rows, write_finalize
from src.sampling.pilot import (draw_pilot, write_pilot_manifest, prevalence_table, prevalence_bounds, rarest,
                                INSTRUCTIONS)
set_seed(42)
TAG = 'T0_2026-10-06'
SOURCE = os.path.join(input_dir('interim'), f'shopee_clean_{TAG}.jsonl')
EXCLUSION = os.path.join(input_dir('manifests'), f'holdout_exclusion_{TAG}.json')
PILOT_MANIFEST = os.path.join(input_dir('manifests'), f'pilot_manifest_{TAG}.json')
PILOT_DIR = os.path.abspath(os.path.join('..', '07_human_eval', 'pilot'))
PILOT_SHEET = os.path.join(PILOT_DIR, 'pilot_blank.csv')

## Rút mẫu pilot

In [2]:
records = load_jsonl(SOURCE)
excluded = json.load(open(EXCLUSION, encoding='utf-8'))['review_ids']
if os.path.exists(PILOT_SHEET) and os.path.exists(PILOT_MANIFEST):
    print('Pilot đã được rút trước đó — giữ nguyên (không ghi đè).')
else:
    pilot_rows, pool_size = draw_pilot(records, n=400, exclude_ids=excluded, seed=42)
    export_blank(pilot_rows, PILOT_SHEET)
    write_pilot_manifest(pilot_rows, pool_size, SOURCE, EXCLUSION, PILOT_SHEET, PILOT_MANIFEST, seed=42)
pilot_manifest = json.load(open(PILOT_MANIFEST, encoding='utf-8'))
pilot_ids = pilot_manifest['review_ids']
print(f"Pool sau khi loại calibration: {pilot_manifest['pool_size']:,} review (corpus {len(records):,} − {len(excluded)})")
print(f'Pilot: {len(pilot_ids)} review → {PILOT_SHEET}')
print('Trùng với calibration:', len(set(pilot_ids) & set(excluded)), '| review_id trùng trong pilot:', len(pilot_ids) - len(set(pilot_ids)))
print()
print(INSTRUCTIONS)

Pilot đã được rút trước đó — giữ nguyên (không ghi đè).
Pool sau khi loại calibration: 55,870 review (corpus 56,050 − 180)
Pilot: 400 review → C:\Workspace\kltn-vn-app-review-requirements\07_human_eval\pilot\pilot_blank.csv
Trùng với calibration: 0 | review_id trùng trong pilot: 0

Pilot prevalence — 400 review ngẫu nhiên, gán bằng guideline v1.2 (FROZEN)
- Nhãn: BUG / FEATURE / UX / OTHER; cột ambiguous: 1 khi ≥2 ý actionable khác lớp không ý nào trội, ngược lại 0; note: lý do khi phân vân.
- Hai người gán độc lập trên hai bản sao Google Sheets riêng (pilot_A / pilot_B), không xem bản của nhau.
- Pilot là một phần của Hold-out: không thảo luận nội dung pilot để sửa guideline; ca lệch được chốt theo guideline v1.2.
- Gán xong tải về pilot_A.csv và pilot_B.csv, đặt trong 07_human_eval/pilot/.


In [3]:
# Mô tả nhanh mẫu pilot so với corpus (chỉ để kiểm tra tính ngẫu nhiên — không dùng nhãn)
by_id = {r['review_id']: r for r in records}
pil = pd.DataFrame([by_id[i] for i in pilot_ids])
cor = pd.DataFrame(records)
print('Phân bố số sao (%) — pilot vs corpus:')
print(pd.DataFrame({'pilot': pil['rating'].value_counts(normalize=True).sort_index() * 100,
                    'corpus': cor['rating'].value_counts(normalize=True).sort_index() * 100}).round(1).to_string())
print('Số từ trung vị — pilot:', int(pil['text_clean'].str.split().str.len().median()),
      '| corpus:', int(cor['text_clean'].str.split().str.len().median()))

Phân bố số sao (%) — pilot vs corpus:
        pilot  corpus
rating               
1        42.8    39.9
2         2.5     3.4
3         3.0     3.2
4         5.2     5.0
5        46.5    48.5
Số từ trung vị — pilot: 7 | corpus: 6


## Đo tỷ lệ nhãn (chạy sau khi hai người gán xong)

Phần này chỉ có kết quả khi đã có `pilot_A.csv` và `pilot_B.csv`. Vì pilot được gán bởi hai người, tôi lấy nhãn đồng thuận; các ca
lệch được xuất ra `pilot_disagreements_to_finalize.csv` để hai người chốt theo guideline v1.2, không tự quyết. Độ đồng thuận trên
pilot cũng là một lần kiểm tra κ bổ sung, lần này trên dữ liệu ngẫu nhiên lớn hơn calibration.

Trước khi chốt các ca lệch, tôi báo tỷ lệ trên phần đồng thuận kèm khoảng chặn dưới/trên (gán mọi ca lệch về phía không thuộc /
thuộc lớp đó). Khi đã có nhãn chốt cho mọi ca lệch, tôi tính tỷ lệ cuối của từng lớp và của nhóm actionable, kèm khoảng tin cậy
95% Wilson — với n = 400 và các lớp hiếm khoảng vài phần trăm, khoảng tin cậy rất quan trọng. Lớp hiếm nhất (p_rare) và khoảng tin
cậy của nó là đầu vào để quyết định kích thước Hold-out ở bước sau.

In [4]:
PA, PB = os.path.join(PILOT_DIR, 'pilot_A.csv'), os.path.join(PILOT_DIR, 'pilot_B.csv')
PILOT_FINALIZE = os.path.join(PILOT_DIR, 'pilot_disagreements_to_finalize.csv')
pilot_df = None
if not (os.path.exists(PA) and os.path.exists(PB)):
    print('Đang chờ pilot_A.csv và pilot_B.csv')
else:
    m = merge(PA, PB, expected_ids=pilot_ids)
    if m['problems']:
        print('Cần sửa tệp gán nhãn trước khi đo:')
        for pr in m['problems']:
            print('   -', pr)
    else:
        pilot_df = m['df']
        r = agreement(pilot_df)
        print(f"Đồng thuận: n={r['n_valid']} | % đồng thuận={r['percent_agreement']} | κ 4 lớp={r['kappa_4class']} "
              f"| κ actionable–OTHER={r['actionable_other']['kappa']} | κ FEATURE–UX={r['feature_ux']['kappa']} "
              f"(n={r['feature_ux']['n_both_feature_or_ux']})")
        print('Ma trận nhầm lẫn (hàng = A, cột = B):')
        print(r['confusion'].to_string())
        groups = disagreements_by_pair(pilot_df)
        print('Số ca lệch:', sum(len(v) for v in groups.values()), '|',
              ', '.join(f'{a}/{b}={len(v)}' for (a, b), v in groups.items()))
        print()
        print('Trước khi chốt ca lệch — tỷ lệ trên phần đồng thuận và khoảng chặn trên toàn pilot:')
        print(pd.DataFrame(prevalence_bounds(pilot_df)).round(4).to_string(index=False))
        if not os.path.exists(PILOT_FINALIZE) and groups:
            write_finalize(build_finalize_rows({'pilot': (PA, PB)}, {}, {}), PILOT_FINALIZE)
            print('Đã xuất ca lệch để chốt nhãn:', PILOT_FINALIZE)

Đồng thuận: n=400 | % đồng thuận=94.2 | κ 4 lớp=0.83 | κ actionable–OTHER=0.865 | κ FEATURE–UX=0.79 (n=38)


Ma trận nhầm lẫn (hàng = A, cột = B):
B        BUG  FEATURE  UX  OTHER
A                               
BUG       29        0   1      6
FEATURE    1        8   1      2
UX         1        2  27      2
OTHER      7        0   0    313
Số ca lệch: 23 | OTHER/BUG=7, BUG/OTHER=6, FEATURE/OTHER=2, UX/FEATURE=2, UX/OTHER=2, FEATURE/BUG=1, FEATURE/UX=1, BUG/UX=1, UX/BUG=1

Trước khi chốt ca lệch — tỷ lệ trên phần đồng thuận và khoảng chặn trên toàn pilot:
     class  consensus_count  share_among_consensus  lower_bound  upper_bound  n_disagreements_touching
       BUG               29                 0.0769       0.0725       0.1125                        16
   FEATURE                8                 0.0212       0.0200       0.0350                         6
        UX               27                 0.0716       0.0675       0.0850                         7
     OTHER              313                 0.8302       0.7825       0.8250                        17
ACTIONABLE               64  

In [5]:
# Tỷ lệ cuối (khi mọi ca lệch đã có agreed_label) + khoảng tin cậy 95% Wilson
if pilot_df is None:
    print('Chưa có dữ liệu gán của pilot.')
else:
    final = {r['review_id']: r['label_a'] for r in pilot_df.to_dict('records') if r['label_a'] == r['label_b']}
    pending = []
    if os.path.exists(PILOT_FINALIZE):
        fin = pd.read_csv(PILOT_FINALIZE, encoding='utf-8-sig', dtype=str).fillna('')
        for r in fin.to_dict('records'):
            lab = r['agreed_label'].strip().upper()
            if lab in ('BUG', 'FEATURE', 'UX', 'OTHER'):
                final[r['review_id']] = lab
            else:
                pending.append(r['review_id'])
    if pending or len(final) != len(pilot_ids):
        print(f'Còn {len(pilot_ids) - len(final)} ca lệch chưa có agreed_label — chưa tính tỷ lệ cuối.')
    else:
        table = prevalence_table(list(final.values()))
        out = pd.DataFrame(table)
        out['tỷ lệ'] = (out['share'] * 100).round(1).astype(str) + '%'
        out['CI 95%'] = out.apply(lambda r: f"{r['ci_low'] * 100:.1f}–{r['ci_high'] * 100:.1f}%", axis=1)
        print(out[['class', 'count', 'tỷ lệ', 'CI 95%']].rename(columns={'class': 'lớp', 'count': 'số ca'}).to_string(index=False))
        rr = rarest(table)
        print(f"\np_rare = {rr['class']}: {rr['share'] * 100:.1f}% (CI 95% {rr['ci_low'] * 100:.1f}–{rr['ci_high'] * 100:.1f}%)")

       lớp  số ca tỷ lệ     CI 95%
       BUG     41 10.2%  7.6–13.6%
   FEATURE     10  2.5%   1.4–4.5%
        UX     31  7.8%  5.5–10.8%
     OTHER    318 79.5% 75.3–83.2%
ACTIONABLE     82 20.5% 16.8–24.7%

p_rare = FEATURE: 2.5% (CI 95% 1.4–4.5%)


## Chốt kích thước Hold-out (amendment v1.7)

Sau khi hai người chốt 23 ca lệch, tỷ lệ trên pilot là BUG 10,2%, FEATURE 2,5% (CI 95% 1,4–4,5%), UX 7,8%, OTHER 79,5% và
actionable 20,5%. FEATURE hiếm hơn tôi nghĩ: theo công thức N ≈ m / p_rare với m = 80, Hold-out cần khoảng 3.200 review, gấp ba
trần 1.000. Nâng trần lên 2.000 trở lên nghĩa là hai người phải gán thêm khoảng 1.600 review mà phần lớn là OTHER, không kịp với
lịch tuần này. Vì vậy chúng tôi giữ Hold-out 1.000 review (400 pilot + 600 rút thêm, ngẫu nhiên thuần) và chấp nhận rằng F1 của
FEATURE trên Hold-out sẽ có khoảng tin cậy rộng — đây là một hạn chế sẽ ghi trong luận văn.

Để bù lại, tôi thêm một **focused set** khoảng 150 review chọn theo các cụm từ hay gặp trong góp ý tính năng và phàn nàn trải nghiệm.
Tập này được hai người gán, khóa như Hold-out và báo cáo riêng, chỉ dùng để xem mô hình tách FEATURE và UX tốt đến đâu; nó không
gộp vào metric chính vì không phải mẫu ngẫu nhiên.

Các tập còn lại: Validation khoảng 350 review rút ngẫu nhiên (chưa có nhãn nên chưa thể stratify; nếu sau khi gán thấy lớp hiếm quá
ít thì bổ sung từ pool còn lại), Train khoảng 1.200 review gồm 180 review calibration cộng phần rút mới có làm giàu lớp hiếm bằng cụm
từ gợi ý — người gán chính làm một mình, người thứ hai gán lại 15% ngẫu nhiên để kiểm tra chất lượng — và Lazada 500 review ngẫu nhiên
làm External Test. Phần xử lý nằm trong `src/sampling/splits.py`.

In [6]:
# Ghi 400 nhãn pilot vào provenance (split = internal_holdout)
from datetime import date
from src.annotation.gold import build_double_label_gold, write_provenance
PROVENANCE = os.path.join(input_dir('gold'), 'provenance.csv')
by_id = {r['review_id']: r for r in records}
prov_now = pd.read_csv(PROVENANCE, encoding='utf-8-sig', dtype=str)
if set(pilot_ids) <= set(prov_now['review_id']):
    print('Nhãn pilot đã có trong provenance — giữ nguyên.')
else:
    pilot_gold, pst = build_double_label_gold('pilot', PA, PB, PILOT_FINALIZE, by_id, sampling_source='random',
                                              split='internal_holdout', annotated_with='v1.2', guideline_version='v1.2',
                                              label_date=date.today().isoformat())
    write_provenance(pilot_gold, PROVENANCE)
    print(f"Ghi {len(pilot_gold)} review pilot | đồng thuận: {pst['consensus']} | đã chốt: {pst['from_finalize']}"
          f" | chỉ một người đánh ambiguous: {len(pst['ambiguous_one_sided'])}")
prov_now = pd.read_csv(PROVENANCE, encoding='utf-8-sig', dtype=str)
print('provenance.csv:', len(prov_now), 'dòng |', prov_now.groupby(['annotation_set', 'split']).size().to_dict())

Ghi 400 review pilot | đồng thuận: 377 | đã chốt: 23 | chỉ một người đánh ambiguous: 2
provenance.csv: 580 dòng | {('batch1', 'train'): 75, ('batch2', 'train'): 75, ('feature_ux', 'train'): 30, ('pilot', 'internal_holdout'): 400}


## Chia tập

In [7]:
from src.annotation.calibration import export_blank
from src.sampling.splits import build_splits, overlap_matrix, write_split_manifest
LZ_SOURCE = os.path.join(input_dir('interim'), f'lazada_clean_{TAG}.jsonl')
SPLITS_MANIFEST = os.path.join(input_dir('manifests'), f'splits_manifest_{TAG}.json')
ANN_DIR = os.path.join(input_dir('gold'), 'annotations')
SIZES = {'holdout_extra': 600, 'validation': 350, 'focused': 150, 'train_extra': 1020,
         'train_cue_share': 0.175, 'train_audit_share': 0.15, 'lazada': 500}
lazada = load_jsonl(LZ_SOURCE)
SHEETS = {'holdout': os.path.join(ANN_DIR, 'holdout_extra_600_blank.csv'),
          'validation': os.path.join(ANN_DIR, 'validation_blank.csv'),
          'focused': os.path.join(ANN_DIR, 'focused_feature_ux_blank.csv'),
          'train': os.path.join(ANN_DIR, 'train_extra_blank.csv'),
          'lazada_external_test': os.path.join(ANN_DIR, 'lazada_blank.csv')}
AUDIT_SHEET = os.path.join(ANN_DIR, 'train_audit_blank.csv')
if os.path.exists(SPLITS_MANIFEST):
    print('Các tập đã được chia trước đó — đọc lại từ manifest, không chia lại.')
    split_manifest = json.load(open(SPLITS_MANIFEST, encoding='utf-8'))
else:
    splits = build_splits(records, lazada, excluded, pilot_ids, SIZES, seed=42)
    overlaps = overlap_matrix(splits, {'calibration': excluded, 'pilot': pilot_ids})
    lz_by_id = {r['review_id']: r for r in lazada}
    for name, s in splits.items():
        to_label = set(s['meta']['to_label'])
        export_blank([r for r in s['rows'] if r['review_id'] in to_label], SHEETS[name])
    audit = set(splits['train']['meta']['audit_ids'])
    export_blank([r for r in splits['train']['rows'] if r['review_id'] in audit], AUDIT_SHEET)
    split_manifest = write_split_manifest(splits, overlaps, {'shopee_interim': SOURCE, 'lazada_interim': LZ_SOURCE},
                                          {**SHEETS, 'train_audit': AUDIT_SHEET}, SPLITS_MANIFEST, seed=42)
print('Manifest:', SPLITS_MANIFEST)

Manifest: C:\Workspace\kltn-vn-app-review-requirements\01_data\manifests\splits_manifest_T0_2026-10-06.json


## Kiểm tra các tập

Tôi kiểm lại ba điều: kích thước và số review còn phải gán của từng tập; không có review nào nằm ở hai tập (chỉ được phép Hold-out
chứa pilot và Train chứa calibration); và phân bố số sao của từng tập so với corpus, để thấy tập nào lấy ngẫu nhiên thì giống corpus,
còn tập làm giàu bằng cụm từ thì lệch đi đến đâu.

In [8]:
sets = split_manifest['sets']
rows = []
for name, s in sets.items():
    rows.append({'tập': name, 'cỡ': s['size'], 'còn phải gán': len(s['to_label']),
                 'cách rút': s['sampling'][:60], 'gán': s['label_mode'][:45], 'khóa': s['locked']})
print(pd.DataFrame(rows).to_string(index=False))
print('Train audit (người thứ hai gán lại):', len(sets['train']['audit_ids']), 'review')
print('Thành phần Train:', sets['train']['composition'])
print('Thành phần focused theo nhóm cụm từ:', sets['focused']['cue_groups'])
print()
print('Giao nhau review_id giữa mọi cặp tập (chỉ được khác 0 ở holdout ∩ pilot = 400 và train ∩ calibration = 180):')
ov = split_manifest['overlaps']
for k, v in ov.items():
    print(f'  {k:42s} {v}')
bad = {k: v for k, v in ov.items() if v and k not in ('holdout ∩ pilot', 'train ∩ calibration')}
assert not bad, f'LEAKAGE: {bad}'
assert ov['holdout ∩ pilot'] == 400 and ov['train ∩ calibration'] == 180
print('→ Không có giao nhau ngoài thiết kế. Calibration chỉ ở Train, pilot chỉ ở Hold-out.')

                 tập   cỡ  còn phải gán                                                     cách rút                                           gán  khóa
             holdout 1000           600                                                  pure random                                        double  True
          validation  350           350 random at draw time; Q8 stratification applied after labelli                                        double False
             focused  150           150                 keyword-enriched, balanced across cue groups                                        double  True
               train 1200          1020 random + keyword enrichment for rare classes (keywords are n single (main annotator) + double-label audit  False
lazada_external_test  500           500                              pure random from Lazada interim                                        double  True
Train audit (người thứ hai gán lại): 153 review
Thành phần Train: {'calibration (l

In [9]:
sh_by_id = {r['review_id']: r for r in records}
lz_by_id = {r['review_id']: r for r in lazada}
dist = {'corpus Shopee': pd.Series([r['rating'] for r in records])}
for name, s in sets.items():
    src = lz_by_id if name == 'lazada_external_test' else sh_by_id
    dist[name] = pd.Series([src[i]['rating'] for i in s['review_ids']])
dist['corpus Lazada'] = pd.Series([r['rating'] for r in lazada])
tab = pd.DataFrame({k: v.value_counts(normalize=True).sort_index() * 100 for k, v in dist.items()}).round(1)
tab.index.name = 'sao'
print('Phân bố số sao (%) theo tập:')
print(tab.to_string())

Phân bố số sao (%) theo tập:
     corpus Shopee  holdout  validation  focused  train  lazada_external_test  corpus Lazada
sao                                                                                         
1             39.9     43.3        35.1     49.3   45.1                  59.0           60.7
2              3.4      3.0         2.3      5.3    4.3                   3.6            3.5
3              3.2      2.7         3.1      8.7    5.6                   3.4            3.8
4              5.0      5.2         6.9     18.0    5.5                   4.4            3.7
5             48.5     45.8        52.6     18.7   39.5                  29.6           28.3
